In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

# ── Load all 63 CIC-IoT-2023 files ───────────────────────────────────────
DATA_PATH = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iot-combined\data\raw\ciciot2023"

files = glob.glob(os.path.join(DATA_PATH, "*.csv"))
print(f"Found {len(files)} files")

df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['Label'].value_counts())

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nMissing values:")
print(df.isnull().sum().sum())
print(f"\nDuplicate rows: {df.duplicated().sum():,}")
print(f"\nData types:")
print(df.dtypes.value_counts())
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# ── Clean and downsample ──────────────────────────────────────────────────

# Step 1: Drop duplicates
df = df.drop_duplicates()
print(f"After dedup: {len(df):,} rows")

# Step 2: Drop missing values
df = df.dropna()
print(f"After dropna: {len(df):,} rows")

# Step 3: Binary label
df['binary_label'] = (df['Label'] != 'BENIGN').astype(int)
print(f"\nBinary label distribution:")
print(df['binary_label'].value_counts())

# Step 4: Downsample to 500k rows stratified
from sklearn.utils import resample

df_sampled = resample(df, n_samples=500000, random_state=42, stratify=df['binary_label'])
print(f"\nDownsampled shape: {df_sampled.shape}")
print(f"Binary label distribution after sampling:")
print(df_sampled['binary_label'].value_counts())

In [ ]:
# ── EDA: Attack distribution in sample ───────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 6))
attack_counts = df_sampled[df_sampled['binary_label'] == 1]['Label'].value_counts()
sns.barplot(x=attack_counts.values, y=attack_counts.index, palette='Purples_r')
plt.title('Attack Type Distribution — CIC-IoT-2023 (500k sample)', fontsize=13)
plt.xlabel('Count')
plt.ylabel('Attack Type')
plt.tight_layout()
plt.savefig(r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iot-combined\results\ciciot2023_attack_dist.png", dpi=150)
plt.show()
print("Saved attack distribution plot")

# Feature summary
print(f"\nFeature stats:")
print(df_sampled.drop(['Label', 'binary_label'], axis=1).describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
import numpy as np

# ── Fix infinite and extreme values ──────────────────────────────────────

# Replace inf with NaN then drop
df_sampled.replace([np.inf, -np.inf], np.nan, inplace=True)
df_sampled.dropna(inplace=True)
print(f"After removing inf: {len(df_sampled):,} rows")

# Check remaining issues
print(f"\nRemaining missing values: {df_sampled.isnull().sum().sum()}")
print(f"Remaining inf values: {np.isinf(df_sampled.select_dtypes(include=[np.number])).sum().sum()}")
print(f"\nFinal shape: {df_sampled.shape}")

In [9]:
# ── Save processed sample ─────────────────────────────────────────────────
import os

processed_path = r"C:\Users\CyrilCorp\PyCharmMiscProject\ids-iot-combined\data\processed"
os.makedirs(processed_path, exist_ok=True)

df_sampled.to_csv(os.path.join(processed_path, "ciciot2023_sampled.csv"), index=False)
print(f"Saved: ciciot2023_sampled.csv")
print(f"Shape: {df_sampled.shape}")
print(f"Binary label distribution:")
print(df_sampled['binary_label'].value_counts())

Saved: ciciot2023_sampled.csv
Shape: (499996, 41)
Binary label distribution:
binary_label
1    475066
0     24930
Name: count, dtype: int64
